In [0]:
path="development_042_silver_sandbox.demand_forecast.igt_everi_attributes_bronze"
dest="development_042_silver_sandbox.demand_forecast.igt_everi_attributes_silver"

In [0]:
df=spark.read.table(path)
df.display()

In [0]:
from pyspark.sql.functions import col, count, when

total_count = df.count()
null_percentages = df.select([
    (count(when(col(c).isNull(), c)) / total_count * 100).alias(c)
    for c in df.columns
])
display(null_percentages)

In [0]:
import re

def clean_column(col_name):
    col_name = col_name.lower().strip()
    # Replace any run of non-alphanumeric characters with a single underscore
    col_name = re.sub(r'[^a-z0-9]+', '_', col_name)
    col_name = col_name.strip('_')  # remove any leading/trailing underscores
    return col_name

new_columns = [clean_column(c) for c in df.columns]
df_cleaned = df.toDF(*new_columns)
display(df_cleaned)

In [0]:
from pyspark.sql.functions import lower
TOLOWER=['theme_name','vendor','product_segment','cabinet','merchandising','key','form_factor','mechanic_segment','brand_evolution','theme','jackpots']
df_lower = df_cleaned.select([lower(col(c)).alias(c) for c in TOLOWER])
display(df_lower)

In [0]:
from pyspark.sql.types import StringType
c=['theme_name','vendor','product_segment','cabinet','merchandising','key','form_factor','mechanic_segment','brand_evolution','theme','jackpots']

string_cols=['jackpots']
for col_name in string_cols:
    vc = df_lower.groupBy(col_name).count().orderBy('count', ascending=False)
    display(vc)

In [0]:
from pyspark.sql.functions import when
mapping={
    'product_segment':{'for sale':'core', 'for sale (core)':'core'}, # for now mlp is as is
    'mechanic_segment':{'proven mech':'proven','unproven mech':'unproven'},# 
    'theme':{'genre nontrending':'non-trending'}
}
df_mapped = df_lower
for col_name, replacements in mapping.items():
    expr = col(col_name)
    for old_val, new_val in replacements.items():
        expr = when(col(col_name) == old_val, new_val).otherwise(expr)
    df_mapped = df_mapped.withColumn(col_name, expr)

display(df_mapped)


In [0]:
df_mapped.write.mode("overwrite").saveAsTable(dest)